# damda — Colab 학습 노트북

Colab T4 GPU 무료 티어에서 damda-ai 학습을 실행하는 노트북.

## 사전 준비

1. **GPU 런타임 활성화**: 상단 `런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU`
2. **Google Drive에 다음 구조로 폴더 업로드**:

```
MyDrive/2026-damda/
├── AI/                    ← 코드 폴더 (로컬 AI/ 전체 업로드)
│   ├── src/
│   ├── configs/
│   ├── notebooks/
│   ├── requirements.txt
│   └── ...
└── data/                  ← 데이터 폴더
    ├── images/
    │   ├── L_CHEEK/
    │   ├── FOREHEAD/
    │   └── ...(부위별 폴더)
    └── labels/
        └── *.json
```

3. **이 노트북을 Colab에서 열기** — `MyDrive/2026-damda/AI/notebooks/colab_train.ipynb`를 더블클릭하면 Colab이 자동 실행.

## 0. GPU 확인

In [ ]:
!nvidia-smi

import torch
print(f"torch={torch.__version__}  CUDA={torch.cuda.is_available()}  device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 1. Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 경로 설정

본인 Drive 구조에 맞게 `PROJECT_ROOT`, `DATA_ROOT` 수정.

In [ ]:
import os, sys

PROJECT_ROOT = '/content/drive/MyDrive/2026-damda/AI'
DATA_ROOT    = '/content/drive/MyDrive/2026-damda/data'

assert os.path.isdir(PROJECT_ROOT), f'PROJECT_ROOT 없음: {PROJECT_ROOT}'
assert os.path.isdir(DATA_ROOT),    f'DATA_ROOT 없음: {DATA_ROOT}'

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATA_ROOT    =', DATA_ROOT)
print('cwd          =', os.getcwd())

## 3. 의존성 설치

Colab은 torch/torchvision/pandas 등 대부분 사전 설치되어 있어 빠르게 끝남.

In [ ]:
!pip install -q -r requirements.txt

## 4. manifest 빌드

AI-Hub JSON을 정규화해 단일 CSV로 만듦. 처음에는 `--limit 200`으로 빠르게 동작 확인.

In [ ]:
IMG_ROOT = f'{DATA_ROOT}/images'
LBL_ROOT = f'{DATA_ROOT}/labels'

!python -m src.build_manifest \
    --image-root "$IMG_ROOT" \
    --json-root  "$LBL_ROOT" \
    --output     data/manifest.csv \
    --limit 200

## 5. 라벨 분포 / 결측률 확인

분류 헤드의 unique 값 개수를 보고 `configs/baseline.yaml`의 `classification_heads`를 조정해야 함.

In [ ]:
import pandas as pd
df = pd.read_csv('data/manifest.csv')
print(f'총 {len(df)}개 샘플\n')

print('=== 부위별 분포 ===')
print(df['region'].value_counts(), '\n')

print('=== 분류 라벨별 unique 값 ===')
for col in ['wrinkle_grade','pigmentation_grade','pore_grade','dryness_grade','sagging_grade','skin_type']:
    if col in df.columns:
        vals = sorted(df[col].dropna().unique())
        print(f'  {col:22s}: {vals}')

print('\n=== 라벨 결측률(%) ===')
print((df.isna().mean() * 100).round(1))

## 6. Baseline Validation 학습

500장 × 10 epoch. 통과 기준 — train loss 감소 / val loss 진동 없음.

T4 GPU에서 ResNet-50 + 500장 × 10 epoch ≈ 5~10분.

In [ ]:
!python -m src.train --config configs/baseline.yaml --validation-mode

## 7. TensorBoard로 학습 곡선 확인

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/

## 8. 본 학습 (Baseline 통과 후)

주석(`#`) 풀고 실행. T4 기준 1만 건 × 30 epoch ≈ 2~4시간 예상.

**Tip**: Colab 무료 티어는 12시간 후 런타임 종료. 장시간 학습은 체크포인트 빈도(`save_every`)를 1~2 epoch으로 줄이고, 연결 해제 시 마지막 체크포인트에서 재개하는 방식을 권장.

In [ ]:
# !python -m src.train --config configs/baseline.yaml

## 9. 체크포인트 / 로그 위치

Drive 위에서 학습했으므로 다음 폴더에 자동 저장됨:
- `checkpoints/epoch*.pt` — 모델 가중치
- `runs/main/` 또는 `runs/baseline_val/` — TensorBoard 로그 + train.log

런타임 해제 후에도 Drive에 그대로 남아있으므로 다음 세션에서 이어서 작업 가능.